# 08 — XGBoost

Trains XGBoost on the Layer A features from `src/features.py` and evaluates every setup on the project's fixed stratified CV splits. The target is ordinal, so the main pipeline predicts a continuous `sii` score and optimizes the three class thresholds directly for quadratic weighted kappa (QWK).

This notebook mirrors notebook 07: classifier baseline, regression and threshold optimization, class weighting, Optuna tuning, Ridge blending, extra features, and a saved results table. All learned preprocessing is fit inside each fold.

In [15]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from scipy.optimize import minimize
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier, XGBRegressor

from src.config import ID_COLUMN, PROCESSED_DIR, PROJECT_ROOT, RANDOM_STATE, RESULTS_DIR, TARGET
from src.evaluation import create_cv_splits, evaluate_model, quadratic_weighted_kappa
from src.imputation import make_preprocessor

experiment_results: list[dict] = []
train_features = pd.read_parquet(PROCESSED_DIR / "train_features.parquet")
print("Train features:", train_features.shape)

Train features: (3960, 67)


## Prepare data and reusable QWK helpers

In [16]:
labeled = train_features[train_features[TARGET].notna()].reset_index(drop=True)
feature_columns = [c for c in labeled.columns if c not in {ID_COLUMN, TARGET}]
X = labeled[feature_columns].copy()
y = labeled[TARGET].astype(int)
ids = labeled[ID_COLUMN]
cv_splits = create_cv_splits(X=X, y=y, ids=ids)
print("X:", X.shape, "| class counts:", y.value_counts().sort_index().to_dict())

def predictions_to_classes(predictions, thresholds):
    return np.digitize(np.asarray(predictions), np.sort(thresholds))

def _negative_qwk(thresholds, predictions, y_true):
    return -cohen_kappa_score(y_true, predictions_to_classes(predictions, thresholds), weights="quadratic")

def optimize_thresholds(predictions, y_true, initial=(0.5, 1.5, 2.5)):
    result = minimize(
        _negative_qwk, np.asarray(initial, dtype=float),
        args=(np.asarray(predictions), np.asarray(y_true)), method="Nelder-Mead",
    )
    return np.sort(result.x)

X: (2736, 65) | class counts: {0: 1594, 1: 730, 2: 378, 3: 34}


## XGBoost classifier baseline

The baseline deliberately treats `sii` as four unordered classes. Median imputation and one-hot encoding are learned independently inside each outer fold. This provides the direct comparison that motivates ordinal regression.

In [ ]:
classifier_pipeline = Pipeline([
    ("preprocessor", make_preprocessor(X, strategy="median", scale=False)),
    ("model", XGBClassifier(
        objective="multi:softprob", num_class=4, eval_metric="mlogloss",
        n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1,
    )),
])
result_classifier = evaluate_model(classifier_pipeline, X, y, cv_splits)
experiment_results.append({
    "setup": "XGBClassifier | Layer A, default-style baseline",
    "mean_val_qwk": np.mean(result_classifier["validation_scores"]),
    "val_std_qwk": np.std(result_classifier["validation_scores"]),
    "mean_train_qwk": np.mean(result_classifier["training_scores"]),
})

Fold 1: training QWK=1.0000, validation QWK=0.2855
Fold 2: training QWK=1.0000, validation QWK=0.3414
Fold 3: training QWK=1.0000, validation QWK=0.3229
Fold 4: training QWK=1.0000, validation QWK=0.2650
Fold 5: training QWK=1.0000, validation QWK=0.3278
Mean training QWK: 1.0000
Mean validation QWK: 0.3085
Validation QWK standard deviation: 0.0286


## Regression + QWK-optimized thresholds

Regression respects the order of the target. For fold-safe early stopping, every outer training fold is split again into a fit portion and a stopping portion. The preprocessor is fit only on the fit portion; neither the stopping data nor the outer validation data influence its statistics.

As in notebook 07, thresholds are optimized once on all OOF predictions. That makes the reported thresholded QWK mildly optimistic; nested threshold fitting is needed for a fully unbiased final estimate.

In [18]:
def fit_predict_oof_regressor(
    build_model, X_frame, y, cv_splits, sample_weight=None,
    early_stopping_fraction=0.15, verbose=True,
):
    oof = np.zeros(len(y), dtype=float)
    fold_scores = []
    for fold, (train_idx, val_idx) in enumerate(cv_splits, start=1):
        X_outer, y_outer = X_frame.iloc[train_idx], y.iloc[train_idx]
        fit_pos, stop_pos = train_test_split(
            np.arange(len(X_outer)), test_size=early_stopping_fraction,
            random_state=RANDOM_STATE, stratify=y_outer,
        )
        preprocessor = make_preprocessor(X_outer.iloc[fit_pos], strategy="median", scale=False)
        X_fit = preprocessor.fit_transform(X_outer.iloc[fit_pos])
        X_stop = preprocessor.transform(X_outer.iloc[stop_pos])
        X_val = preprocessor.transform(X_frame.iloc[val_idx])
        model = build_model()
        fit_kwargs = {"eval_set": [(X_stop, y_outer.iloc[stop_pos])], "verbose": False}
        if sample_weight is not None:
            fit_kwargs["sample_weight"] = np.asarray(sample_weight)[train_idx][fit_pos]
        model.fit(X_fit, y_outer.iloc[fit_pos], **fit_kwargs)
        oof[val_idx] = model.predict(X_val)
        fold_score = quadratic_weighted_kappa(y.iloc[val_idx], predictions_to_classes(oof[val_idx], [0.5, 1.5, 2.5]))
        fold_scores.append(fold_score)
        if verbose:
            print(f"Fold {fold}: fixed-threshold validation QWK={fold_score:.4f}, best iteration={model.best_iteration}")
    return oof, fold_scores

def build_default_regressor():
    return XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
        learning_rate=0.03, max_depth=6, early_stopping_rounds=100,
        random_state=RANDOM_STATE, n_jobs=-1, tree_method="hist",
    )

In [19]:
oof_default, default_fold_scores = fit_predict_oof_regressor(build_default_regressor, X, y, cv_splits)
fixed_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_default, [0.5, 1.5, 2.5]))
thresholds_default = optimize_thresholds(oof_default, y)
default_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_default, thresholds_default))
print("Fixed thresholds QWK:", round(fixed_qwk, 4))
print("Optimized thresholds:", thresholds_default.round(3), "-> QWK:", round(default_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | Layer A, default params + optimized thresholds",
    "mean_val_qwk": default_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

Fold 1: fixed-threshold validation QWK=0.3420, best iteration=127
Fold 2: fixed-threshold validation QWK=0.3768, best iteration=67
Fold 3: fixed-threshold validation QWK=0.3485, best iteration=55
Fold 4: fixed-threshold validation QWK=0.3471, best iteration=90
Fold 5: fixed-threshold validation QWK=0.3202, best iteration=64
Fixed thresholds QWK: 0.347
Optimized thresholds: [0.606 0.921 2.752] -> QWK: 0.4287


## Class imbalance

Test inverse-square-root class weights on the regressor. QWK already penalizes distant ordinal mistakes, so weighting is measured rather than assumed to help.

In [20]:
class_counts = y.value_counts()
class_weight_map = (1.0 / np.sqrt(class_counts)).to_dict()
sample_weight = y.map(class_weight_map).to_numpy()
sample_weight = sample_weight / sample_weight.mean()
oof_weighted, _ = fit_predict_oof_regressor(
    build_default_regressor, X, y, cv_splits, sample_weight=sample_weight,
)
thresholds_weighted = optimize_thresholds(oof_weighted, y)
weighted_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_weighted, thresholds_weighted))
print("Weighted regressor QWK:", round(weighted_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | class-weighted + optimized thresholds",
    "mean_val_qwk": weighted_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

Fold 1: fixed-threshold validation QWK=0.3227, best iteration=179
Fold 2: fixed-threshold validation QWK=0.3876, best iteration=146
Fold 3: fixed-threshold validation QWK=0.3817, best iteration=104
Fold 4: fixed-threshold validation QWK=0.3467, best iteration=440
Fold 5: fixed-threshold validation QWK=0.3537, best iteration=321
Weighted regressor QWK: 0.4182


## Optuna hyperparameter tuning

Tune tree capacity, sampling, and regularization against OOF QWK itself. Early stopping still uses only an inner holdout. Forty trials matches notebook 07; increase the budget after the pipeline has been validated end to end.

In [21]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.55, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 30.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }
    def build():
        return XGBRegressor(
            objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
            early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
            tree_method="hist", **params,
        )
    oof, _ = fit_predict_oof_regressor(build, X, y, cv_splits, verbose=False)
    thresholds = optimize_thresholds(oof, y)
    return quadratic_weighted_kappa(y, predictions_to_classes(oof, thresholds))

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=40, show_progress_bar=True)
print("Best tuning QWK:", round(study.best_value, 4))
print("Best params:", study.best_params)

Best trial: 35. Best value: 0.472226: 100%|██████████| 40/40 [02:24<00:00,  3.62s/it]

Best tuning QWK: 0.4722
Best params: {'learning_rate': 0.014850259678316026, 'max_depth': 5, 'min_child_weight': 1.413347955356425, 'subsample': 0.8762511234570196, 'colsample_bytree': 0.5605936678257406, 'reg_alpha': 3.726158311196111, 'reg_lambda': 23.07622742870599, 'gamma': 1.7853520582128581}


In [22]:
def build_tuned_regressor():
    return XGBRegressor(
        objective="reg:squarederror", eval_metric="rmse", n_estimators=3000,
        early_stopping_rounds=100, random_state=RANDOM_STATE, n_jobs=-1,
        tree_method="hist", **study.best_params,
    )

oof_tuned, _ = fit_predict_oof_regressor(build_tuned_regressor, X, y, cv_splits)
thresholds_tuned = optimize_thresholds(oof_tuned, y)
tuned_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_tuned, thresholds_tuned))
print("Tuned QWK:", round(tuned_qwk, 4), "| thresholds:", thresholds_tuned.round(3))
experiment_results.append({
    "setup": "XGBRegressor | Optuna-tuned + optimized thresholds",
    "mean_val_qwk": tuned_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

Fold 1: fixed-threshold validation QWK=0.3494, best iteration=502
Fold 2: fixed-threshold validation QWK=0.4117, best iteration=385
Fold 3: fixed-threshold validation QWK=0.3640, best iteration=395
Fold 4: fixed-threshold validation QWK=0.3276, best iteration=895
Fold 5: fixed-threshold validation QWK=0.3368, best iteration=478
Tuned QWK: 0.4722 | thresholds: [0.606 0.934 2.704]


## Blend tuned XGBoost with Ridge

In [23]:
oof_ridge = np.zeros(len(y), dtype=float)
for train_idx, val_idx in cv_splits:
    ridge = Pipeline([
        ("preprocessor", make_preprocessor(X.iloc[train_idx], strategy="median")),
        ("model", Ridge(alpha=1.0)),
    ])
    ridge.fit(X.iloc[train_idx], y.iloc[train_idx])
    oof_ridge[val_idx] = ridge.predict(X.iloc[val_idx])
thresholds_ridge = optimize_thresholds(oof_ridge, y)
ridge_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_ridge, thresholds_ridge))
experiment_results.append({
    "setup": "Ridge | Layer A reference + optimized thresholds",
    "mean_val_qwk": ridge_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

best_weight, best_blend_qwk, best_blend_thresholds = None, -np.inf, None
for xgb_weight in np.arange(0.0, 1.01, 0.05):
    blended = xgb_weight * oof_tuned + (1.0 - xgb_weight) * oof_ridge
    thresholds = optimize_thresholds(blended, y)
    score = quadratic_weighted_kappa(y, predictions_to_classes(blended, thresholds))
    if score > best_blend_qwk:
        best_weight, best_blend_qwk, best_blend_thresholds = xgb_weight, score, thresholds
print(f"Best blend: {best_weight:.2f} XGBoost + {1-best_weight:.2f} Ridge -> QWK={best_blend_qwk:.4f}")
experiment_results.append({
    "setup": "Blend | tuned XGBRegressor + Ridge + optimized thresholds",
    "mean_val_qwk": best_blend_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

Best blend: 1.00 XGBoost + 0.00 Ridge -> QWK=0.4722


## Extra engineered features

Repeat the tuned model with the same experimental domain features used in notebook 07. They remain local to this notebook until CV demonstrates that they help.

In [24]:
train_extra = train_features.copy()
train_extra["Fitness_Endurance_total_seconds"] = train_extra["Fitness_Endurance-Time_Mins"] * 60 + train_extra["Fitness_Endurance-Time_Sec"]
train_extra["Physical-Waist_to_Height"] = train_extra["Physical-Waist_Circumference"] / train_extra["Physical-Height"]
train_extra["Physical-Pulse_Pressure"] = train_extra["Physical-Systolic_BP"] - train_extra["Physical-Diastolic_BP"]
train_extra["FGC_total_score"] = train_extra[[
    "FGC-FGC_CU", "FGC-FGC_GSND", "FGC-FGC_GSD", "FGC-FGC_PU",
    "FGC-FGC_SRL", "FGC-FGC_SRR", "FGC-FGC_TL",
]].sum(axis=1, skipna=True)
train_extra["BMI_per_Age"] = train_extra["Physical-BMI"] / train_extra["Basic_Demos-Age"]
labeled_extra = train_extra[train_extra[TARGET].notna()].reset_index(drop=True)
X_extra = labeled_extra[[c for c in labeled_extra.columns if c not in {ID_COLUMN, TARGET}]].copy()
oof_extra, _ = fit_predict_oof_regressor(build_tuned_regressor, X_extra, y, cv_splits)
thresholds_extra = optimize_thresholds(oof_extra, y)
extra_qwk = quadratic_weighted_kappa(y, predictions_to_classes(oof_extra, thresholds_extra))
print("Tuned + extra features QWK:", round(extra_qwk, 4), "| delta:", round(extra_qwk-tuned_qwk, 4))
experiment_results.append({
    "setup": "XGBRegressor | tuned + extra features + optimized thresholds",
    "mean_val_qwk": extra_qwk, "val_std_qwk": np.nan, "mean_train_qwk": np.nan,
})

Fold 1: fixed-threshold validation QWK=0.3380, best iteration=502
Fold 2: fixed-threshold validation QWK=0.4136, best iteration=386
Fold 3: fixed-threshold validation QWK=0.3600, best iteration=628
Fold 4: fixed-threshold validation QWK=0.3352, best iteration=988
Fold 5: fixed-threshold validation QWK=0.3397, best iteration=478
Tuned + extra features QWK: 0.4767 | delta: 0.0044


## Summary and saved results

In [25]:
summary = pd.DataFrame(experiment_results).round(4).sort_values("mean_val_qwk", ascending=False)
RESULTS_DIR.mkdir(exist_ok=True)
results_path = RESULTS_DIR / "xgboost_cv_results.csv"
summary.to_csv(results_path, index=False)
print("Saved results to:", results_path.relative_to(PROJECT_ROOT))
summary

Saved results to: results\xgboost_cv_results.csv


,setup,mean_val_qwk,val_std_qwk,mean_train_qwk
6,XGBRegressor | tuned + extra features + optimi...,0.4767,NaN,NaN
3,XGBRegressor | Optuna-tuned + optimized thresh...,0.4722,NaN,NaN
5,Blend | tuned XGBRegressor + Ridge + optimized...,0.4722,NaN,NaN
4,Ridge | Layer A reference + optimized thresholds,0.4410,NaN,NaN
1,"XGBRegressor | Layer A, default params + optim...",0.4287,NaN,NaN
2,XGBRegressor | class-weighted + optimized thre...,0.4182,NaN,NaN
0,"XGBClassifier | Layer A, default-style baseline",0.3085,0.0286,1.0


## Reading the result

Use the sorted table rather than assuming the most complex setup wins. In particular, check whether class weighting, the Ridge blend, and extra features beat the plain tuned regressor. For a final model, rerun threshold selection inside a nested CV loop or validate the chosen thresholds on a separate holdout before fitting all labeled rows and predicting the test set.